# Music Recommendation System — Dataset Generation
### Approach
This notebook generates our self-compiled **World Dataset** (`CSV` file) containing 126 songs (well within the 200-song limit) across 18 artists and 9 genres.
* **Column 1 (`Song_Name`)**: Formatted as `"Song Title - Artist"`.
* **Columns 2 to 19 (Artist Columns)**: One-hot binary encoded (`1` if the song is by that artist, `0` otherwise).
* **Columns 20 to 28 (Genre Columns)**: One-hot binary encoded (`1` if the song belongs to that genre, `0` otherwise).

In [1]:
import pandas as pd
import numpy as np

CSV_FILENAME = "Data.csv"

# 1. Define Artists, Songs, and Base Genres (126 songs total)
artists_catalog = {
    "Queen": {
        "songs": [
            "Bohemian Rhapsody", "Don't Stop Me Now", "Another One Bites the Dust",
            "We Will Rock You", "Somebody to Love", "Killer Queen", "Under Pressure", "Radio Ga Ga"
        ],
        "genres": ["Classic Rock", "Hard Rock", "Pop"]
    },
    "ABBA": {
        "songs": [
            "Dancing Queen", "Take a Chance on Me", "Mamma Mia", "Gimme! Gimme! Gimme!",
            "The Winner Takes It All", "Waterloo", "Lay All Your Love on Me", "SOS"
        ],
        "genres": ["Pop", "Dance Pop"]
    },
    "Britney Spears": {
        "songs": [
            "Toxic", "Piece of Me", "...Baby One More Time", "Oops!... I Did It Again",
            "Gimme More", "Womanizer", "Circus", "Everytime"
        ],
        "genres": ["Pop", "Dance Pop"]
    },
    "The Weeknd": {
        "songs": [
            "Call Out My Name", "Blinding Lights", "Starboy", "The Hills",
            "Save Your Tears", "Can't Feel My Face", "After Hours", "Die For You"
        ],
        "genres": ["R&B", "Alt R&B", "Pop"]
    },
    "Eagles": {
        "songs": [
            "Hotel California", "Take It Easy", "Desperado", "Life in the Fast Lane",
            "One of These Nights", "New Kid in Town", "Tequila Sunrise"
        ],
        "genres": ["Classic Rock"]
    },
    "Nirvana": {
        "songs": [
            "Smells Like Teen Spirit", "Come as You Are", "Lithium",
            "Heart-Shaped Box", "In Bloom", "About a Girl", "All Apologies"
        ],
        "genres": ["Grunge / Alt Rock", "Hard Rock"]
    },
    "Beyoncé": {
        "songs": [
            "Halo", "Crazy in Love", "Single Ladies", "Cuff It",
            "Formation", "Irreplaceable", "Love on Top", "Break My Soul"
        ],
        "genres": ["R&B", "Pop", "Dance Pop"]
    },
    "Daft Punk": {
        "songs": [
            "One More Time", "Get Lucky", "Harder, Better, Faster, Stronger",
            "Around the World", "Instant Crush", "Digital Love", "Da Funk"
        ],
        "genres": ["Electronic/EDM", "Dance Pop"]
    },
    "Led Zeppelin": {
        "songs": [
            "Stairway to Heaven", "Immigrant Song", "Whole Lotta Love",
            "Black Dog", "Kashmir", "Rock and Roll", "Ramble On"
        ],
        "genres": ["Classic Rock", "Hard Rock"]
    },
    "Metallica": {
        "songs": [
            "Enter Sandman", "Master of Puppets", "Nothing Else Matters",
            "One", "For Whom the Bell Tolls", "Sad But True", "Fade to Black"
        ],
        "genres": ["Metal", "Hard Rock"]
    },
    "Linkin Park": {
        "songs": [
            "In the End", "Numb", "Crawling", "Faint",
            "Somewhere I Belong", "Breaking the Habit", "What I've Done"
        ],
        "genres": ["Grunge / Alt Rock", "Hard Rock", "Metal"]
    },
    "Michael Jackson": {
        "songs": [
            "Billie Jean", "Beat It", "Thriller", "Smooth Criminal",
            "Man in the Mirror", "Black or White", "Rock with You"
        ],
        "genres": ["Pop", "Dance Pop", "R&B"]
    },
    "Pink Floyd": {
        "songs": [
            "Comfortably Numb", "Wish You Were Here", "Another Brick in the Wall",
            "Time", "Money", "Shine On You Crazy Diamond"
        ],
        "genres": ["Classic Rock"]
    },
    "Rihanna": {
        "songs": [
            "Umbrella", "Diamonds", "We Found Love", "Only Girl (In the World)",
            "Stay", "Needed Me", "Work"
        ],
        "genres": ["Pop", "R&B", "Dance Pop"]
    },
    "Calvin Harris": {
        "songs": [
            "Summer", "Feel So Close", "This Is What You Came For",
            "One Kiss", "Sweet Nothing", "Outside"
        ],
        "genres": ["Electronic/EDM", "Dance Pop"]
    },
    "Frank Ocean": {
        "songs": [
            "Thinkin Bout You", "Nights", "Pink + White", "Ivy", "Lost", "Chanel"
        ],
        "genres": ["R&B", "Alt R&B"]
    },
    "Black Sabbath": {
        "songs": [
            "Paranoid", "Iron Man", "War Pigs", "N.I.B.",
            "Children of the Grave", "Heaven and Hell"
        ],
        "genres": ["Metal", "Hard Rock", "Classic Rock"]
    },
    "Pearl Jam": {
        "songs": [
            "Alive", "Even Flow", "Jeremy", "Black", "Yellow Ledbetter", "Better Man"
        ],
        "genres": ["Grunge / Alt Rock", "Hard Rock"]
    }
}

# 2. Song-specific genre nuances so tracks by the same artist aren't all identical vectors
song_genre_overrides = {
    "Another One Bites the Dust - Queen": ["Classic Rock", "Dance Pop", "Pop"],
    "Under Pressure - Queen": ["Classic Rock", "Pop"],
    "Radio Ga Ga - Queen": ["Classic Rock", "Pop", "Electronic/EDM"],
    "Blinding Lights - The Weeknd": ["R&B", "Pop", "Dance Pop", "Electronic/EDM"],
    "Starboy - The Weeknd": ["R&B", "Alt R&B", "Pop", "Electronic/EDM"],
    "Can't Feel My Face - The Weeknd": ["R&B", "Pop", "Dance Pop"],
    "Beat It - Michael Jackson": ["Pop", "Hard Rock", "R&B"],
    "Nothing Else Matters - Metallica": ["Metal", "Hard Rock", "Classic Rock"],
    "Break My Soul - Beyoncé": ["Pop", "Dance Pop", "Electronic/EDM", "R&B"],
    "Halo - Beyoncé": ["Pop", "R&B"],
    "Instant Crush - Daft Punk": ["Electronic/EDM", "Pop", "Grunge / Alt Rock"],
    "Get Lucky - Daft Punk": ["Electronic/EDM", "Dance Pop", "Pop"],
    "We Found Love - Rihanna": ["Pop", "Dance Pop", "Electronic/EDM"],
    "Needed Me - Rihanna": ["R&B", "Alt R&B"],
    "Stay - Rihanna": ["Pop", "R&B"],
    "Everytime - Britney Spears": ["Pop"],
    "Toxic - Britney Spears": ["Pop", "Dance Pop", "Electronic/EDM"],
    "All Apologies - Nirvana": ["Grunge / Alt Rock", "Classic Rock"],
    "Stairway to Heaven - Led Zeppelin": ["Classic Rock"],
    "Breaking the Habit - Linkin Park": ["Grunge / Alt Rock", "Electronic/EDM"]
}

all_artists = list(artists_catalog.keys())
all_genres = [
    "Classic Rock", "Hard Rock", "Metal", "Grunge / Alt Rock",
    "Pop", "Dance Pop", "R&B", "Alt R&B", "Electronic/EDM"
]

# 3. Construct binary one-hot encoded rows
dataset_rows = []
for artist, data in artists_catalog.items():
    for song in data["songs"]:
        full_song_name = f"{song} - {artist}"
        active_genres = song_genre_overrides.get(full_song_name, data["genres"])

        row = {"Song_Name": full_song_name}
        for a in all_artists:
            row[a] = 1 if a == artist else 0
        for g in all_genres:
            row[g] = 1 if g in active_genres else 0

        dataset_rows.append(row)

# 4. Save to CSV and trigger download to your computer
world_df = pd.DataFrame(dataset_rows)
world_df.to_csv(CSV_FILENAME, index=False)

print(f"Created '{CSV_FILENAME}' with {world_df.shape[0]} songs and {world_df.shape[1]} columns.")
display(world_df.head(5))

from google.colab import files
files.download(CSV_FILENAME)

Created 'Data.csv' with 126 songs and 28 columns.


,Song_Name,Queen,ABBA,Britney Spears,The Weeknd,Eagles,Nirvana,Beyoncé,Daft Punk,Led Zeppelin,...,Pearl Jam,Classic Rock,Hard Rock,Metal,Grunge / Alt Rock,Pop,Dance Pop,R&B,Alt R&B,Electronic/EDM
0,Bohemian Rhapsody - Queen,1,0,0,0,0,0,0,0,0,...,0,1,1,0,0,1,0,0,0,0
1,Don't Stop Me Now - Queen,1,0,0,0,0,0,0,0,0,...,0,1,1,0,0,1,0,0,0,0
2,Another One Bites the Dust - Queen,1,0,0,0,0,0,0,0,0,...,0,1,0,0,0,1,1,0,0,0
3,We Will Rock You - Queen,1,0,0,0,0,0,0,0,0,...,0,1,1,0,0,1,0,0,0,0
4,Somebody to Love - Queen,1,0,0,0,0,0,0,0,0,...,0,1,1,0,0,1,0,0,0,0


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>